<a href="https://colab.research.google.com/github/zarduya/SQL-DB-PROJECT/blob/main/sql_database.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ayrus Pharmacy: Randomly Generated Retail Database
This notebook generates a realistic synthetic SQLite database for a fictional UK
health and beauty retailer, for my 7PAM2005 Data Mining & Discovery SQL assignment.
All data is randomly generated; no real customer data is used.

In [11]:
!pip install faker


In [12]:
import sqlite3
import random
import numpy as np
import pandas as pd
from faker import Faker

In [13]:
# set the seed so the random data comes out identical every time we run the notebook
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
fake = Faker('en_GB')   # for british names
Faker.seed(SEED)

In [14]:
from datetime import date

# some fixed variables ?
BRAND = "Ayrus Pharmacy"
OWN_BRAND = "Ayrus"
N_PRODUCTS = 250
DATA_START = date(2025, 1, 1)
DATA_END = date(2025, 12, 31)

# regions get their own table so that each name is typed only once
regions_df = pd.DataFrame({
    'region_id': [1, 2, 3, 4, 5],
    'region_name': ['London', 'South East', 'East of England',
                    'West Midlands', 'North West']
})
regions_df

,region_id,region_name
0,1,London
1,2,South East
2,3,East of England
3,4,West Midlands
4,5,North West


In [15]:
# stores are typed in manually since a real company already knows where its stores are
# format is ordinal: Local < High Street < Flagship
STORE_LIST = [
    ('Oxford Street', 'London', 1, 'Flagship'),
    ('Croydon', 'London', 1, 'High Street'),
    ('Stratford', 'London', 1, 'High Street'),
    ('Brighton', 'Brighton', 2, 'High Street'),
    ('Reading', 'Reading', 2, 'High Street'),
    ('Oxford', 'Oxford', 2, 'Local'),
    ('Luton', 'Luton', 3, 'High Street'),
    ('Cambridge', 'Cambridge', 3, 'High Street'),
    ('Watford', 'Watford', 3, 'Local'),
    ('Norwich', 'Norwich', 3, 'Local'),
    ('Birmingham Bullring', 'Birmingham', 4, 'Flagship'),
    ('Coventry', 'Coventry', 4, 'Local'),
    ('Manchester Arndale', 'Manchester', 5, 'Flagship'),
    ('Liverpool', 'Liverpool', 5, 'High Street'),
    ('Preston', 'Preston', 5, 'Local'),
]

# typical floor size for each format as (average, spread) in m²
# normal distribution so stores of the same format are similar but not identical
SIZE_BY_FORMAT = {'Local': (150, 30), 'High Street': (400, 80), 'Flagship': (1200, 200)}

stores = []
for store_id, (label, city, region_id, fmt) in enumerate(STORE_LIST, start=1):
    mean, sd = SIZE_BY_FORMAT[fmt]
    size = round(max(np.random.normal(mean, sd), 50), 1)   # never smaller than 50 m²
    opened = fake.date_between(start_date=date(1985, 1, 1), end_date=date(2022, 12, 31))
    stores.append({
        'store_id': store_id,
        'store_name': f"{BRAND} {label}",
        'city': city,
        'region_id': region_id,            # foreign key to regions
        'store_format': fmt,               # ordinal
        'size_sq_m': size,                 # ratio, 0 m² would mean no space at all
        'opened_date': opened.isoformat()  # interval, saved as 'YYYY-MM-DD' text since SQLite has no date type
    })

stores_df = pd.DataFrame(stores)
stores_df

,store_id,store_name,city,region_id,store_format,size_sq_m,opened_date
0,1,Ayrus Pharmacy Oxford Street,London,1,Flagship,1299.3,2009-04-18
1,2,Ayrus Pharmacy Croydon,London,1,High Street,388.9,1985-12-14
2,3,Ayrus Pharmacy Stratford,London,1,High Street,451.8,1995-06-14
3,4,Ayrus Pharmacy Brighton,Brighton,2,High Street,521.8,1993-06-25
4,5,Ayrus Pharmacy Reading,Reading,2,High Street,381.3,2012-12-25
5,6,Ayrus Pharmacy Oxford,Oxford,2,Local,143.0,2010-09-18
6,7,Ayrus Pharmacy Luton,Luton,3,High Street,526.3,2018-11-25
7,8,Ayrus Pharmacy Cambridge,Cambridge,3,High Street,461.4,1988-04-21
8,9,Ayrus Pharmacy Watford,Watford,3,Local,135.9,2001-01-12
9,10,Ayrus Pharmacy Norwich,Norwich,3,Local,166.3,1986-02-18


In [16]:
# each category has its own brands, items and sizes so the product names make sense
# real brand names are used to look real, but all prices and sales are made up
# share = how much of the 250 products are in this category
# median_price = normal price for the smallest size, in £
CATEGORIES = {
    'Skincare': {'brands': ['CeraVe', 'Nivea', 'La Roche-Posay', 'Garnier'],
                 'items': ['Moisturising Cream', 'Hydrating Cleanser', 'Vitamin C Serum', 'Night Cream', 'SPF 50 Sun Cream'],
                 'sizes': ['50ml', '100ml', '200ml'], 'median_price': 12.0, 'share': 0.20, 'own_brand': True},
    'Haircare': {'brands': ['Pantene', 'TRESemmé', 'Aussie', 'Herbal Essences'],
                 'items': ['Shampoo', 'Conditioner', 'Hair Mask', 'Dry Shampoo'],
                 'sizes': ['250ml', '400ml', '500ml'], 'median_price': 4.5, 'share': 0.14, 'own_brand': True},
    'Vitamins': {'brands': ['Seven Seas', 'Centrum', 'Vitabiotics'],
                 'items': ['Multivitamin Tablets', 'Vitamin D 25µg', 'Vitamin C 1000mg', 'Omega-3 Capsules', 'Iron Tablets'],
                 'sizes': ['30 pack', '60 pack', '90 pack'], 'median_price': 7.0, 'share': 0.14, 'own_brand': True},
    'Pain Relief': {'brands': ['Nurofen', 'Panadol', 'Anadin'],
                 'items': ['Tablets', 'Caplets', 'Extra Strength Tablets', 'Liquid Capsules'],
                 'sizes': ['8 pack', '16 pack', '32 pack'], 'median_price': 3.0, 'share': 0.10, 'own_brand': True},
    'Cold & Flu': {'brands': ['Lemsip', 'Beechams'],
                 'items': ['Max Strength Capsules', 'Hot Lemon Sachets', 'Day & Night Capsules', 'Cold & Flu Tablets'],
                 'sizes': ['5 pack', '10 pack', '16 pack'], 'median_price': 5.0, 'share': 0.10, 'own_brand': True},
    'Oral Care': {'brands': ['Colgate', 'Sensodyne', 'Aquafresh'],
                 'items': ['Whitening Toothpaste', 'Sensitive Toothpaste', 'Total Care Toothpaste', 'Fresh Mint Mouthwash', 'Alcohol-Free Mouthwash'],
                 'sizes': ['Single', 'Twin Pack'], 'median_price': 4.0, 'share': 0.08, 'own_brand': True},
    # no own brand perfume, pharmacies don't usually make their own perfume
    'Fragrance': {'brands': ['Hugo Boss', 'Calvin Klein', 'Paco Rabanne', 'Jimmy Choo'],
                 'items': ['Eau de Toilette', 'Eau de Parfum', 'Body Spray'],
                 'sizes': ['30ml', '50ml', '100ml'], 'median_price': 45.0, 'share': 0.12, 'own_brand': False},
    'Baby':     {'brands': ['Pampers', 'Huggies'],
                 'items': ['Nappies Size 2', 'Nappies Size 3', 'Nappies Size 4', 'Nappies Size 5', 'Sensitive Baby Wipes'],
                 'sizes': ['Single Pack', 'Value Pack', 'Jumbo Pack'], 'median_price': 9.0, 'share': 0.12, 'own_brand': True},
}

In [17]:
products = []
seen_names = set()    # remembers names already made so we don't get the same product twice
base_prices = {}      # one price per product, so all its sizes are priced in line
product_id = 1

# bigger sizes cost more, but cheaper per unit, like in real shops
SIZE_MULTIPLIERS = [1.0, 1.6, 2.2]

for category, info in CATEGORIES.items():
    n_in_category = round(N_PRODUCTS * info['share'])
    brands = info['brands'] + ([OWN_BRAND] if info['own_brand'] else [])
    created = 0

    while created < n_in_category:
        brand = random.choice(brands)
        item = random.choice(info['items'])
        size = random.choice(info['sizes'])
        name = f"{brand} {item} {size}"
        if name in seen_names:
            continue              # already made, try again
        seen_names.add(name)

        # pick the price once per product
        # log-normal means most things are cheap and only a few are expensive
        key = (brand, item)
        if key not in base_prices:
            base = np.random.lognormal(mean=np.log(info['median_price']), sigma=0.3)
            if brand == OWN_BRAND:
                base *= 0.7                      # own brand is about 30% cheaper
            base_prices[key] = base

        price = base_prices[key] * SIZE_MULTIPLIERS[info['sizes'].index(size)]
        price = max(round(price) - 0.01, 0.99)   # make prices look like £4.99

        products.append({
            'product_id': product_id,
            'product_name': name,
            'category': category,          # nominal
            'brand': brand,                # nominal
            'unit_price': price,           # ratio
            'is_own_brand': int(brand == OWN_BRAND)   # 1 = own brand, 0 = not
        })
        product_id += 1
        created += 1

products_df = pd.DataFrame(products)
print(len(products_df), "products generated")
products_df.sort_values('product_name').head(15)

250 products generated


,product_id,product_name,category,brand,unit_price,is_own_brand
139,140,Anadin Extra Strength Tablets 8 pack,Pain Relief,Anadin,2.99,0
126,127,Anadin Liquid Capsules 16 pack,Pain Relief,Anadin,3.99,0
144,145,Anadin Liquid Capsules 32 pack,Pain Relief,Anadin,5.99,0
132,133,Anadin Tablets 16 pack,Pain Relief,Anadin,3.99,0
141,142,Anadin Tablets 32 pack,Pain Relief,Anadin,5.99,0
185,186,Aquafresh Alcohol-Free Mouthwash Single,Oral Care,Aquafresh,3.99,0
188,189,Aquafresh Fresh Mint Mouthwash Single,Oral Care,Aquafresh,3.99,0
182,183,Aquafresh Fresh Mint Mouthwash Twin Pack,Oral Care,Aquafresh,6.99,0
187,188,Aquafresh Sensitive Toothpaste Twin Pack,Oral Care,Aquafresh,5.99,0
189,190,Aquafresh Total Care Toothpaste Single,Oral Care,Aquafresh,3.99,0


In [18]:
# check the products look right
print(products_df['category'].value_counts(), "\n")
print(products_df.groupby('category')['unit_price'].median().round(2), "\n")
print(products_df['unit_price'].describe().round(2))

category
Skincare       50
Haircare       35
Vitamins       35
Baby           30
Fragrance      30
Pain Relief    25
Cold & Flu     25
Oral Care      20
Name: count, dtype: int64 

category
Baby           11.99
Cold & Flu      6.99
Fragrance      69.49
Haircare        5.99
Oral Care       5.49
Pain Relief     3.99
Skincare       15.99
Vitamins       10.99
Name: unit_price, dtype: float64 

count    250.00
mean      17.98
std       24.54
min        1.99
25%        5.99
50%        9.99
75%       16.99
max      138.99
Name: unit_price, dtype: float64


# customers

In [19]:
N_CUSTOMERS = 1175   # real customers, about 25 copies get added later to make around 1,200

# more women than men shop at pharmacies, so the weights show that
GENDERS = ['Female', 'Male', 'Non-binary', 'Prefer not to say']
GENDER_WEIGHTS = [0.62, 0.34, 0.01, 0.03]

# most people are Bronze, only a few reach Platinum
TIERS = ['Bronze', 'Silver', 'Gold', 'Platinum']
TIER_WEIGHTS = [0.55, 0.28, 0.13, 0.04]

EMAIL_DOMAINS = ['gmail.com', 'outlook.com', 'hotmail.co.uk', 'yahoo.co.uk', 'icloud.com']
DOMAIN_WEIGHTS = [0.45, 0.20, 0.15, 0.10, 0.10]

# bigger stores get more customers, but square root stops the big ones getting too many
store_weights = np.sqrt(stores_df['size_sq_m'])
store_weights = store_weights / store_weights.sum()

def make_email(first, last, used_emails):
    """make an email from the person's name so name and email match"""
    f = ''.join(c for c in first.lower() if c.isalpha())   # remove things like ' and -
    l = ''.join(c for c in last.lower() if c.isalpha())
    pattern = random.choice(['{f}.{l}', '{f}{l}{n}', '{fi}{l}', '{f}_{l}{n}'])
    local = pattern.format(f=f, l=l, fi=f[0], n=random.randint(1, 99))
    domain = random.choices(EMAIL_DOMAINS, DOMAIN_WEIGHTS)[0]
    email = f"{local}@{domain}"
    while email in used_emails:            # if two people have the same name, add a number
        email = f"{local}{random.randint(100, 999)}@{domain}"
    used_emails.add(email)
    return email

In [20]:
customers = []
used_emails = set()   # remembers emails already used so none repeat

for customer_id in range(1, N_CUSTOMERS + 1):
    gender = random.choices(GENDERS, GENDER_WEIGHTS)[0]
    # first name matches gender so it looks real
    if gender == 'Female':
        first = fake.first_name_female()
    elif gender == 'Male':
        first = fake.first_name_male()
    else:
        first = fake.first_name()
    last = fake.last_name()

    # most people are around 43, kept between 16 and 90
    age = int(np.clip(np.random.normal(43, 16), 16, 90))

    # picking the customer's home store, bigger stores get picked more
    store_id = int(np.random.choice(stores_df['store_id'], p=store_weights))
    # customer lives in the same city as their store
    city = stores_df.loc[stores_df['store_id'] == store_id, 'city'].iloc[0]

    customers.append({
        'customer_id': customer_id,
        'first_name': first,                                   # nominal
        'last_name': last,                                     # nominal
        'email': make_email(first, last, used_emails),         # nominal, every email is different
        'phone': f"07{random.randint(100000000, 999999999)}",  # UK mobile style number
        'age': age,                                            # ratio
        'gender': gender,                                      # nominal
        'city': city,                                          # nominal
        'preferred_store_id': store_id,                        # foreign key to stores
        'loyalty_tier': random.choices(TIERS, TIER_WEIGHTS)[0],  # ordinal
        'signup_date': fake.date_between(date(2016, 1, 1), date(2025, 10, 31)).isoformat()  # interval
    })

customers_df = pd.DataFrame(customers)
print(len(customers_df), "customers generated")
customers_df.sample(10)

1175 customers generated


,customer_id,first_name,last_name,email,phone,age,gender,city,preferred_store_id,loyalty_tier,signup_date
570,571,Harriet,Freeman,harrietfreeman93@hotmail.co.uk,07586909856,62,Female,Manchester,13,Silver,2024-10-05
803,804,Sally,Brown,sally_brown56@outlook.com,07467435349,48,Female,Birmingham,11,Bronze,2018-01-13
695,696,Denise,Marshall,denise.marshall@icloud.com,07459048149,40,Female,Luton,7,Bronze,2022-04-12
409,410,Carly,O'Sullivan,carly.osullivan@gmail.com,07254460628,53,Female,Cambridge,8,Silver,2017-07-09
272,273,Suzanne,Bradley,suzannebradley63@outlook.com,07844567935,30,Female,Watford,9,Gold,2021-05-29
1171,1172,Jacqueline,Smith,jacqueline.smith@gmail.com,07537309736,39,Female,Brighton,4,Silver,2020-06-27
1157,1158,Karen,Byrne,karen.byrne@yahoo.co.uk,07331478831,32,Female,London,3,Bronze,2024-08-22
343,344,Shannon,Hill,shannon.hill@hotmail.co.uk,07240346956,22,Female,London,1,Bronze,2024-12-24
232,233,Valerie,Thomas,valerie_thomas11@hotmail.co.uk,07417161302,52,Female,Manchester,13,Silver,2016-07-14
1112,1113,Jordan,Haynes,jhaynes832@yahoo.co.uk,07805403426,81,Male,London,1,Bronze,2025-09-08


In [21]:
# check the customers look right
print(customers_df['gender'].value_counts(), "\n")
print(customers_df['loyalty_tier'].value_counts(), "\n")
print(customers_df['age'].describe().round(1), "\n")
print(customers_df['city'].value_counts(), "\n")
print("All emails unique:", customers_df['email'].is_unique)

gender
Female               695
Male                 411
Prefer not to say     56
Non-binary            13
Name: count, dtype: int64 

loyalty_tier
Bronze      652
Silver      328
Gold        145
Platinum     50
Name: count, dtype: int64 

count    1175.0
mean       43.5
std        15.1
min        16.0
25%        33.0
50%        43.0
75%        53.0
max        90.0
Name: age, dtype: float64 

city
London        308
Manchester    143
Birmingham    121
Luton          93
Brighton       85
Reading        72
Cambridge      71
Liverpool      61
Norwich        51
Coventry       49
Watford        48
Oxford         40
Preston        33
Name: count, dtype: int64 

All emails unique: True
